In [ ]:
# ==========================================
# ENVIRONMENT SETUP & DRIVE MOUNTING
# ==========================================
import os
import sys
import time
import itertools
import numpy as np
import pandas as pd
import cv2
import PIL.Image as im
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, Flatten, Dense, MaxPool2D, 
                                     MaxPooling2D, AveragePooling2D, 
                                     Activation, Dropout, BatchNormalization, Input)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

# Mount Google Drive for dataset access
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ==========================================
# 1. VIDEO FRAMING & 2. FACE EXTRACTION
# ==========================================

# Path initialization
location = '/content/drive/MyDrive/Database/'
face_cascade = cv2.CascadeClassifier('/content/drive/MyDrive/Haar Cascade/haarcascade_frontalface_default.xml')

# Part A: Processing frames from directory
test_filenames = filter(lambda x: x.endswith('.jpg'), os.listdir('/content/drive/MyDrive/Untitled folder/'))
paths_to_test_images = ['/content/drive/MyDrive/Untitled folder/' + x for x in test_filenames]

count = 0
withface = 0
withoutface = 0
totalfaces = 0

start = time.time()
for data_path in paths_to_test_images:
    img = cv2.imread(data_path)
    imgframe = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    
    for (x, y, w, h) in faces:
        cv2.rectangle(imgframe, (x, y), (x+w, y+h), (255, 0, 0), 2)
        roi_color = imgframe[y:y + h, x:x + w]
        totalfaces += 1
        
    if len(faces) > 0:
        withface += 1
        cv2.imwrite('/content/drive/MyDrive/test/img_{}.jpg'.format(count), roi_color)
    else:
        withoutface += 1
        cv2.imwrite('/content/drive/MyDrive/test/img_{}.jpg'.format(count), roi_color)
    count += 1
end = time.time()

# Part B: ISED Video Frame Extraction
dirs = os.listdir(location)
dirs = [directory for directory in dirs if os.path.isdir(os.path.join(location, directory))]

frame_class = {}
for i, current_dir in enumerate(dirs):
    files = os.listdir(os.path.join(location, current_dir))
    files = [vid for vid in files if '.avi' in vid]
    frame_class[current_dir] = i + 1
    
    for vid in files:
        video = cv2.VideoCapture(os.path.join(location, current_dir, vid))
        v_count = 0
        success = True
        
        txt_path = os.path.join(location, vid[2:].split('.avi')[0]) + '.txt'
        with open(txt_path, 'w') as f:
            while success:
                success, image = video.read()
                if v_count % 5 == 0 and success:
                    out_path = os.path.join(location, current_dir, vid.split('.avi')[0] + "_frame%d.jpg" % v_count)
                    cv2.imwrite(out_path, image)
                    f.write(os.path.join(current_dir, vid.split('.avi')[0] + "_frame%d.jpg %d\n" % (v_count, i+1)))
                v_count += 1
        video.release()

In [ ]:
# ==========================================
# DATA LOADING CONFIGURATION
# ==========================================
train_data_path = "/content/drive/MyDrive/Dataset/Train"
test_data_path = "/content/drive/MyDrive/Dataset/Test"
valid_data_path = "/content/drive/MyDrive/Dataset/Validation"

# Global hyperparameters (Adjust batch_size to 100 here for step 3 of findings)
batch_size = 10 
epochs = 25

In [ ]:
# ==========================================
# 3. IMPLEMENTATION OF LENET
# ==========================================
img_rows, img_cols = 28, 28

train_datagen = ImageDataGenerator()
train_generator_lenet = train_datagen.flow_from_directory(train_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=True)

valid_datagen = ImageDataGenerator()
valid_generator_lenet = valid_datagen.flow_from_directory(valid_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=True)

test_datagen = ImageDataGenerator()
test_generator_lenet = test_datagen.flow_from_directory(test_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=False)

# LeNet Construction
lenet_model = Sequential([
    Conv2D(filters=6, kernel_size=(5,5), strides=(1,1), activation='tanh', input_shape=(28,28,3), padding='same'),
    AveragePooling2D(),
    Conv2D(filters=16, kernel_size=(5,5), strides=(1,1), activation='tanh', padding='valid'),
    AveragePooling2D(),
    Flatten(),
    Dense(120, activation='tanh'),
    Dense(84, activation='tanh'),
    Dense(4, activation='softmax')
])

lenet_model.summary()

# Compiled with RMSprop as per Findings #2
lenet_model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001), metrics=['accuracy'])

# Train Model
lenet_history = lenet_model.fit(
    train_generator_lenet, 
    steps_per_epoch=int(1111/batch_size), 
    validation_data=valid_generator_lenet, 
    validation_steps=int(729/batch_size), 
    epochs=epochs, 
    verbose=1
)

lenet_score = lenet_model.evaluate(test_generator_lenet)
print("[INFO] LeNet Accuracy: {:.2f}%".format(lenet_score[1] * 100))
print("[INFO] LeNet Loss: ", lenet_score[0])

In [ ]:
# ==========================================
# 4. IMPLEMENTATION OF ALEXNET
# ==========================================
img_rows, img_cols = 227, 227

train_generator_alexnet = train_datagen.flow_from_directory(train_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=True)
valid_generator_alexnet = valid_datagen.flow_from_directory(valid_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=True)
test_generator_alexnet = test_datagen.flow_from_directory(test_data_path, target_size=(img_rows, img_cols), batch_size=batch_size, class_mode='categorical', shuffle=False)

# AlexNet Construction
alexnet_model = Sequential([
    Conv2D(filters=96, kernel_size=(11,11), strides=(4,4), activation='relu', input_shape=(227,227,3)),
    BatchNormalization(),
    MaxPool2D(pool_size=(3,3), strides=(2,2)),
    Conv2D(filters=256, kernel_size=(5,5), strides=(1,1), activation='relu', padding="same"),
    BatchNormalization(),
    MaxPool2D(pool_size=(3,3), strides=(2,2)),
    Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), activation='relu', padding="same"),
    BatchNormalization(),
    Conv2D(filters=384, kernel_size=(1,1), strides=(1,1), activation='relu', padding="same"),
    BatchNormalization(),
    Conv2D(filters=256, kernel_size=(1,1), strides=(1,1), activation='relu', padding="same"),
    BatchNormalization(),
    MaxPool2D(pool_size=(3,3), strides=(2,2)),
    Flatten(),
    Dense(4096, activation='relu'),
    Dropout(0.5),
    Dense(4096, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

alexnet_model.summary()

# Compiled with SGD as per Findings #1 & #2
alexnet_model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.SGD(learning_rate=0.001), metrics=['accuracy'])

# Train Model
alexnet_history = alexnet_model.fit(
    train_generator_alexnet, 
    steps_per_epoch=int(1111/batch_size), 
    validation_data=valid_generator_alexnet, 
    validation_steps=int(729/batch_size), 
    epochs=epochs, 
    verbose=1
)

alexnet_score = alexnet_model.evaluate(test_generator_alexnet)
print("[INFO] AlexNet Accuracy: {:.2f}%".format(alexnet_score[1] * 100))
print("[INFO] AlexNet Loss: ", alexnet_score[0])

In [ ]:
# ==========================================
# 5. PLOT ACCURACY/LOSS & 6. CONFUSION MATRIX
# ==========================================

def plot_performance(history, model_name="Model"):
    # Plot Loss
    plt.figure(figsize=[8,6])
    plt.plot(history.history['loss'], 'r', linewidth=3.0)
    plt.plot(history.history['val_loss'], 'b', linewidth=3.0)
    plt.legend(['Training loss', 'Validation Loss'], fontsize=14)
    plt.xlabel('Epochs ', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title(f'{model_name} Loss Curves', fontsize=14)
    plt.show()

    # Plot Accuracy
    plt.figure(figsize=[8,6])
    plt.plot(history.history['accuracy'], 'r', linewidth=3.0)
    plt.plot(history.history['val_accuracy'], 'b', linewidth=3.0)
    plt.legend(['Training Accuracy', 'Validation Accuracy'], fontsize=14)
    plt.xlabel('Epochs ', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title(f'{model_name} Accuracy Curves', fontsize=14)
    plt.show()

def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(8,8))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=14)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm = np.around(cm, decimals=2)
        print("Normalized confusion matrix")
        
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j], horizontalalignment="center", color="white" if cm[i, j] > thresh else "black")
        
    plt.tight_layout()
    plt.ylabel('True label', fontsize=12)
    plt.xlabel('Predicted label', fontsize=12)
    plt.show()

def evaluate_predictions(model, test_generator, train_generator):
    target_names = [key for key in train_generator.class_indices.keys()]
    
    Y_pred = model.predict(test_generator)
    y_pred = np.argmax(Y_pred, axis=1)
    
    print('\n--- Confusion Matrix ---')
    cm = confusion_matrix(test_generator.classes, y_pred)
    plot_confusion_matrix(cm, target_names, title='Confusion Matrix Extraction')
    
    print('\n--- Classification Report ---')
    print(classification_report(test_generator.classes, y_pred, target_names=target_names))

# Run analysis for AlexNet as an example (Can change parameters to lenet assets as needed)
plot_performance(alexnet_history, "AlexNet")
evaluate_predictions(alexnet_model, test_generator_alexnet, train_generator_alexnet)